This notebook shows minimal end-to-end examples for setting up an AI client in `aidu`. The client is typically a thin abstraction layer over an underlying `AI API` and can target a local or remote AI service.

# Use a classical local AI service

To avoid requiring an OpenAI API key, the first example uses `SymPyClient`. This is a classical AI workflow, so no LLM API access is needed yet. Everything runs on your local machine.

> **How to work with Jupyter**
>
> 1. Run the cells from top to bottom.
> 2. Update the `problem` string in Step 4 if you want to try another expression.
> 3. Inspect the parsed JSON output in Step 5.

The default example computes:

- Derivative of `7x^2 + 3x - 5` with respect to `x`.

## Step 1: Import global packages

We use `os` to interact with the operating system, like reading files. `json` is used to parse structured content returned by the agent. `time` helps us to measure how long things take. The packages `dotenv` is used to load environment variables from a `.env` file, which is useful for storing sensitive information like API keys. The `rich` package is used to print results in a pretty readable format. `IPython.display` contains display helpers.

In [ ]:
import os
import json
import time

from dotenv import load_dotenv

from rich.console import Console
from IPython.display import Math


## Step 2: Import local packages

### Various supporting utilities

Python and its packages are very powerful and give us a lot of tools to work with. Nevertheless, we have implemented some helper functions to make our life easier. These are located in the support package, which we import here.

In [16]:
from aidu.support.filesystem.search import find_up



### Context management

As we want to use the same calling pattern regardless of which client we use, we introduce a thin abstraction layer that lives in the `client` package.

If you have worked with `ChatGPT`, you have probably already heard about the concept of prompting a model. Interacting with a model is not just about sending a message, but also about providing the right definition of the task.

When working with the API, you have even more control over the interaction, although the concept remains similar. Instead of exchanging only simple messages and answers, the interaction is built from a richer context that may include system prompts, conversation traces, state information, tool results, and control parameters. The model processes this contextual information at every interaction turn to generate its response.

This is where the concept of `Context` comes in. It contains the first message — the system message — which defines the prompt and uses the special role `system`. After that, additional messages can have the role `user` or `assistant`, representing the ongoing conversation between the user and the model.

As we will see in more complex scenarios, additional parameters can also be added for each interaction turn. All the information that accompanies the user message is referred to as the `Context` of the interaction.

Building reliable AI applications requires a structured way to manage the context that is sent to and received from the model. This context is typically **not just the entire conversation history**. For better results, it is usually a carefully consolidated representation of the interaction history, combined with the **current state of the interaction** and a set of **control parameters** used to steer the model’s behavior.

In our framework, we collect all this information in a `Context` object, which contains:

* the `Trace` of messages,
* the `State` of the interaction,
* and the `Control` parameters used to steer the model’s behavior.

This structured approach provides a clear and consistent way to manage information flow in AI applications.

The local client is defined in `clients.sympy`.


In [17]:
from aidu.ai.llm.client import Context
from aidu.ai.llm.client import Trace

### Calling AI models

To interact with an AI model — either local or remote — we use a `Client` object. The client acts as a lightweight abstraction layer over the underlying communication interface, whether it is implemented as a local function call, an HTTP request, or another transport mechanism.

Its responsibility is to send the user input together with the current `Context` to the model, receive the generated response, and normalize the result into a structured representation that can be handled consistently across the framework.


In [18]:
import aidu.ai.llm.clients.sympy as sp  # Local client for symbolic math
import aidu.ai.llm.clients.openai as openai_llm  # Remote client for OpenAI models (e.g. GPT-4 etc.)

For our first usage, we create a local math client `sp.SymPyClient` that returns JSON output. Before that, we initialize a rich console, to be ready for readable pretty printing.


In [19]:
console = Console()
client = sp.SymPyClient(
    "sympy",
    config={"enforce_json": True},
    process=sp.solve_math_problem_with_sympy,
)

## Step 4: Define the math problem

Like all other clients, this math client expects a user message as input. The actual user input is represented as an math expression string stored in the `content` field of the message. Together with the role `user`, this forms the user message that is sent to the model.


In [20]:
# Step 4: Define the math problem and create the message
problem = "diff(7x^2 + 3x - 5, x)"
message = {"role": "user", "content": problem}

## Step 5: Run the request and inspect the result

Following our standard calling pattern, we create a `Context` object — which is empty for the moment — and send it together with the user message to the client. The client processes the request and returns a structured response that we can inspect and process further.


In [21]:
context = Context()
response = client.chat(message, context)
content = json.loads(response["content"])

console.print(content, width=80)

{
    'type': 'derivative',
    'expression': '7*x**2 + 3*x - 5',
    'result': '14*x + 3',
    'latex': '14x+3',
    'message': 'When we differentiate $7x^{2}+3x-5$, we get $14x+3$ from SymPy.'
}

# Use an LLM service

In a second step, we can now try to solve the same problem using a language model. In this case, however, we interact with a remote service, which typically requires an API key for authentication.


> **API key required**
>
> Get your **API key from the service provider** -here OpenAI- and define it in `.env` before running any cells:
>
> `OPENAI_API_KEY=your_api_key_here`

## Step 1: Load API key and import the LLM client

With a properly configured .env file, we can load the environment variables and retrieve `OPENAI_API_KEY` to access the OpenAI-backed client.

In [22]:
env_path = find_up(".env")
print(f"Loading environment variables from: {env_path}")
load_dotenv(env_path)


api_key = os.getenv("OPENAI_API_PHBERN_KEY")
assert api_key, "Missing OPENAI_API_PHBERN_KEY in .env"

Loading environment variables from: /home/wspahn/.env


## Step 2: Create the LLM client and run the same task

The generation of the AI client mirrors the classical example above. We create an LLM client, enforcing json output to be ready to request the answer from the OpenAI service. 

In [23]:
gpt_4o_mini_client = openai_llm.OpenAIClient(
    "gpt-4o-mini",
    config={"enforce_json": True},
    api_key=api_key,
)

## Step3: Create the prompt

In contrast to the approach above, where the result was defined by a local Python function, we now interact with a remote LLM service with very general capabilities. As a consequence, we must define the task explicitly using natural language instructions. This is the well-known concept of prompting a model. The prompt is defined in the system message, which is part of the context sent to the model. The system message contains instructions that guide the model to generate the desired output. In this case, we instruct the model to solve a math problem and return the result in JSON format.

In [24]:
prompt = (
    "You are a deterministic math solver like a sympy solver. "
    "Solve the given math problem and return strict JSON with exactly these keys: "
    "type, expression, result, latex, message. "
    "Do not add extra keys or markdown."
)

## Step 4: Create the Context and run the request

The chat call now carries two important pieces of information: the user `message` or request — which we reuse from above — and the non-empty `Context`, which provides the system prompt and the additional information required for the interaction.

The LLM responds with a `dict` containing the generated answer under the `content` field, together with additional metadata such as token usage. Following our instructions, the model encodes its output in JSON format.


In [25]:
system_message = {
    "role": "system",
    "content": prompt,
}

context = Context(trace=Trace(messages=[system_message]))

llm_response = gpt_4o_mini_client.chat(message, context)

console.print(llm_response)

{
    'content': '{\n  "type": "calculation",\n  "expression": "diff(7x^2 + 3x - 5, x)",\n  "result": "14x + 3",\n  
"latex": "14x + 3",\n  "message": "The derivative of the expression 7x^2 + 3x - 5 with respect to x is 14x + 
3."\n}',
    'role': 'assistant',
    'prompt_tokens': 71,
    'completion_tokens': 87,
    'total_tokens': 158,
    'cost_usd': 6.285e-05,
    'model': 'gpt-4o-mini'
}

Following our instructions, the content returned by the LLM is formatted as JSON, but it is still encoded as a string. We therefore parse the JSON response and pretty-print the resulting data structure.


In [26]:
llm_content = json.loads(llm_response["content"])

console.print(llm_content)

{
    'type': 'calculation',
    'expression': 'diff(7x^2 + 3x - 5, x)',
    'result': '14x + 3',
    'latex': '14x + 3',
    'message': 'The derivative of the expression 7x^2 + 3x - 5 with respect to x is 14x + 3.'
}

# Finding Zeros

Now lets see where are the limits of the LLM. First we build a test equation.

## Using SymPy

With the SymPy client, we can easily build a more complex problem, like finding the zeros of a function. By defining the factorial normal form of the function, we know what zeros to expect. Expanding the function gives us a more complex form of the same function, where the zeros are not immediately obvious, which is harder to solve. 

In [27]:
from sympy import symbols, expand, sympify, latex

x = symbols("x")

equation = "(sqrt(3)-x)*(4+x)*(3.5-x)"

expr = sympify(equation)

display(Math(r"\text{factored: }\quad " + latex(expr)))

result = expand(expr)

display(Math(r"\text{expanded: }\quad " + latex(result)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Feeding the expanded form to the SymPy client, we can still find the zeros without any issues. This demonstrates the power of symbolic computation in handling complex expressions. It is fast and accurate, making it an excellent tool for solving mathematical problems, and testing the limits of LLMs.

In [28]:
message = {"role": "user", "content": f"solve({result}, x)"}

# timed llm execution
start = time.perf_counter()
response = client.chat(message, Context())
end = time.perf_counter()

content = json.loads(response["content"])

console.print(f"Sympy call took {end - start:.3f} seconds")
display(Math(r"\text{Zeros: }\quad " + content["latex"]))

Sympy call took 0.103 seconds

<IPython.core.display.Math object>

## Using GPT 4o mini

GPT 4a mini is a good compromise between speed and precision, but sadly not in math. When we feed the same problem to the LLM, it fails to find the correct zeros. Sometimes it is close, but not exact. Sometimes it does not even emit proper math. Rerun the cell a few times and you will see that the results are not consistent. We will see later, that even here the LLM can be helpful, when we guide and supervise it properly, but in this case it fails to solve the problem on its own.

This highlights the limitations of current LLMs in handling complex mathematical problems, especially when they require symbolic manipulation and precise calculations.

In [29]:
system_message = {
    "role": "system",
    "content": prompt,
}

context = Context(trace=Trace(messages=[system_message]))

console.print(f"[bold cyan]Prompt for GPT 4o mini:[/bold cyan] {context.trace.messages[0]['content']}")

# timed llm execution
start = time.perf_counter()
llm_response = gpt_4o_mini_client.chat(message, context)
end = time.perf_counter()

llm_content = json.loads(llm_response["content"])

console.print(f"Used tokens: {llm_response['total_tokens']} for {float(llm_response['cost_usd']) * 1000.0:.2f} ¢")


console.print(f"LLM call took {end - start:.3f} seconds")

display(Math(r"\text{Zeros: }\quad " + llm_content["latex"]))

Prompt for GPT 4o mini: You are a deterministic math solver like a sympy solver. Solve the given math problem and 
return strict JSON with exactly these keys: type, expression, result, latex, message. Do not add extra keys or 
markdown.

Used tokens: 280 for 0.12 ¢

LLM call took 4.961 seconds

<IPython.core.display.Math object>

## Using GPT 5 mini

Now we use the flagship model `gpt-5`. We can reuse the context from previous interactions. It can do proper math, even its mini variant, but it needs **many tokens**, it is **very slow** (800 times slower than sympy locally) and **costly** (40 times the cost of GPT 4o mini), and we have to assume the model is offloading math to cumputer algebra helpers, but it has to think a lot.

In [ ]:
gpt_5_mini_client = openai_llm.OpenAIClient(
    "gpt-5-mini",
    config={"enforce_json": True},
    api_key=api_key,
)

console.print(f"[bold cyan]Prompt for GPT 5 mini:[/bold cyan] {context.trace.messages[0]['content']}")

if False:  # Set to True once
    # timed llm execution
    start = time.perf_counter()
    llm_response = gpt_5_mini_client.chat(message, context)
    end = time.perf_counter()

    console.print(f"Used tokens: {llm_response['total_tokens']} for {float(llm_response['cost_usd']) * 1000.0:.2f} ¢")

    llm_content = json.loads(llm_response["content"])

    console.print(f"LLM call took {end - start:.3f} seconds")
    display(Math(r"\text{expanded: }\quad " + llm_content["latex"]))
else:
    console.print("⚠️ You need to allow this code explicitly as it is expensive.You will see the right solution with Tokens: 2076, COsts 4 ¢, Time: 17 sec.")


Prompt for GPT 5 mini: You are a deterministic math solver like a sympy solver. Solve the given math problem and 
return strict JSON with exactly these keys: type, expression, result, latex, message. Do not add extra keys or 
markdown.

⚠️ You need to allow this code explicitly as it is expensive.You will see the right solution with Tokens: 2076, 
COsts 4 ¢, Time: 17 sec.

## Using Gemini

We can also try a Google model, which is 2x faster than GPT 5 mini, but still much slower than SymPy. It typically does not even try to solve the problem, but just outputs the original equation somewhat reformated in LaTeX format, which is not what we wanted.

In [31]:
import aidu.ai.llm.clients.google as google_llm

google_api_key = os.getenv("GOOGLE_API_KEY")
gemini_2_5_flash_lite = google_llm.GoogleClient(
    "gemini-2.5-flash-lite",
    config={"enforce_json": True},
    api_key=google_api_key,
)

# show the prompt for the next LLM call
console.print(f"[bold cyan]Prompt for GPT 5 mini:[/bold cyan] {context.trace.messages[0]['content']}")

# timed llm execution
start = time.perf_counter()
llm_response = gemini_2_5_flash_lite.chat(message, context)
end = time.perf_counter()

# console.print(llm_response)

Prompt for GPT 5 mini: You are a deterministic math solver like a sympy solver. Solve the given math problem and 
return strict JSON with exactly these keys: type, expression, result, latex, message. Do not add extra keys or 
markdown.

Here we can try even multiiple times, but the result is always the same. This shows that the model is not even trying to solve the problem, but just parroting back the input in a different format. But at least this is done consistently.

This highlights the importance of choosing the right tool for the job, some are more suited for certain tasks than others.

In [32]:
llm_content = json.loads(llm_response["content"])

console.print(f"Used tokens: {llm_response['total_tokens']} for {float(llm_response['cost_usd']) * 1000.0:.2f} ¢")


console.print(f"LLM call took {end - start:.3f} seconds")

display(Math(r"\text{Zeros: }\quad " + llm_content["latex"]))

Used tokens: 281 for 0.08 ¢

LLM call took 7.835 seconds

<IPython.core.display.Math object>